# Following Andrej Karpathy's Course on a Bigram Name Generator    

Excercises suggested in the video description:

<span style = "color:green;"> E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model? </span>

E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

E06: meta-exercise! Think of a fun/interesting exercise and complete it.

First Load in the names into the 'words' list

In [1]:
words = open('names.txt', 'r').read().splitlines()

Now we want to assign an index for each character within a word, going from a = 1, b = 2, to ...., z = 26. We will define the start and end of a word to have the character '.' which we assign the index of '.' = 0. 
We then define how to move from the index to character and the reverse with itos and stoi.

In [2]:
import torch

chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
itos = {i: s for s, i in stoi.items()} # creates a reverse mapping from integer indices back to characters.


Now we create the inputs and labels which the neural network will take. For each word in the 'words' list, we break it up into character plus start and end points, zip these characters up into sequential pairs and index, then append these to the input and label lists. These lists are then converted to pyTorch tensors for the simple Neural Network. 

This is creating the training data set for the Neural Network.

In [3]:
# These will hold the inputs (xs - the current character) and targets (ys - the next character) for training the bigram model.
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.'] # converts each word into a list of characters with '.' at the start and end.
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1] # gets the integer index for the current character
        ix2 = stoi[ch2] # gets the integer index for the next character
        xs.append(ix1) # appends the current character index to the input list
        ys.append(ix2) # appends the next character index to the target list

xs = torch.tensor(xs) # converts the input list to a PyTorch tensor
ys = torch.tensor(ys) # converts the target list to a PyTorch tensor

Now we initialize the weights of the network, which are 27 x 27 in shape for the 27 x 27 possible combinations of characters. 

In [4]:
# creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
g = torch.Generator().manual_seed(2147483647) 

# Weights for the bigram model, initialized randomly. 
# These weights can also be set to zero initially as a smoothing technique if desired.
# 27x27 weight matrix for bigram probabilities (26 letters + '.')
W = torch.randn((27, 27), generator=g, requires_grad=True) 

Now we actually run the model, applying gradient descent. This means we loop over the following process for a predefined number of iterations:
- complete the forward pass. This means determining the probability of the next character given the current character, and subsequently the loss that we aim to minimise.

- assign zero gradient to each weight (if you don't do this then the gradients will accumulate across iterations).

- apply backpropagation to find the gradient of the loss function with respect to each weight.

- update the weight values accordingly to minimise the loss function.

In [6]:
import torch.nn.functional as F

for k in range(100):

    # ===== Forward pass =====
    # Convert the input indices into a 27 dimensional one-hot encoded vectors.
    # The position corresponding each the character index is set to 1 and all other positions are set to 0.
    # It is required to convert to floats for the matrix multiplication with the weights.
    xenc = F.one_hot(xs, num_classes=27).float() # converts the input indices to one-hot encoded vectors
    
    # Computes the logits by multiplying the one-hot encoded input with the weight matrix
    # This is essentially the dot product between xenc and the corresponding row in W for each input character.
    logits = xenc @ W # computes the logits by multiplying the one-hot encoded input with the weight matrix

    # To work out the probabilities, we convert the logits to something resembling counts via exponentiation. 
    counts = logits.exp() # converts the logits to counts by exponentiating them

    # We can then convert these counts to probabilities by normalizing them (dividing by the sum of counts for each input character).
    probs = counts / counts.sum(1, keepdim=True) # normalizes the counts to get probabilities

    # ===== Loss =====
    # We can now compute the loss using negative log likelihood.
    # We select the probabilities corresponding to the target characters (ys) and take the negative log of those probabilities.
    # Loss is defined as the average negative log likelihood across all training examples.
    # The second term is a regularization term that penalizes large weights to prevent overfitting.
    loss = -probs[torch.arange(len(xs)), ys].log().mean() + 0.01 * (W**2).mean() # computes the negative log likelihood loss

    # ===== Backward pass =====
    # We can compute the gradients of the loss with respect to the weights using backpropagation
    W.grad = None # clears the gradients from the previous iteration
    loss.backward() # computes the gradients of the loss with respect to the weights

    # ===== Update weights =====
    # We can update the weights using gradient descent.
    # We subtract a small multiple of the gradient from the weights to move in the direction that reduces the loss.
    # The value of the coefficient (learning rate) determines how big of a step we take in the direction of the negative gradient.
    # This can be tweaked to find the optimal learning rate for training the model effectively.
    W.data += -50 * W.grad # updates the weights using gradient descent 



Printing out the final loss function to check that the gradient descent has worked. From the implicit calculation using the counts in the dataset as shown by Karpathy, we know that we should expect a loss function of around 2.5.

In [7]:
print(loss.item()) # prints the final loss after training

2.4829957485198975


Once this has been run for an appriopriate number of iterations such that the loss is sufficiently minimised, we can generate a sample from the Neural Network using the final weights and the identical forward pass.

In [8]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5): #number of names we wish to generate
    out=[]
    ix = 0 # start with the '.' character
    while True:
        # === Forward pass to get probabilities for the next character ===
        # This is unchanged from the training loop except we are working with a single character index (ix)
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float() # one-hot encode the current character index
        logits = xenc @ W # compute the logits for the next character
        counts = logits.exp() # convert logits to counts
        probs = counts / counts.sum(1, keepdim=True) # normalize counts to get probabilities

        # Sample the next character index from the probability distribution
        # The torch.multinomial function is a prebuilt function used to sample from the probability distribution.
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix]) # append the corresponding character to the output list
        
        if ix == 0: # if we reach the '.' character, we stop generating characters for this name
            break
    print(''.join(out)) # join the list of characters to form a name and print

cexze.
momasurailezityha.
konimittain.
llayn.
ka.


The final output is pretty bad, but this is largely due to the Bigram model being terrible. Now it is time to try the exercises suggested by Karpathy and implement a Trigram Model - a model that uses the two preceding characters to predict the third. 

## Trigram Character Generation

First we create the training data set as before. 
I have stored the two preceding characters in a tuple in xs - maybe there is a better way to do this I don't know.
In order to account for starting without a seed character, we have to include two '.' characters at the start of every word rather than just one. 

In [ ]:
# Now xs will contain the indices of the current characters and ys will contain the indices of the next character.
xs, ys = [], []

for w in words:
    chars =  ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chars, chars[1:], chars[2:]):
        # gets the integer indices for each of the three characters
        ix1 = stoi[ch1] 
        ix2 = stoi[ch2] 
        ix3 = stoi[ch3] 
        # appends the tuple of the first and second character indices to the input list
        # appends the third integer index for the third character to the target list
        xs.append((ix1, ix2)) 
        ys.append(ix3) 

# converting to PyTorch tensors for training
xs = torch.tensor(xs) 
ys = torch.tensor(ys) 


(torch.Size([228146, 2]), torch.Size([228146]))

Now we need to initialise the weights as before, but for a larger xs.


In [110]:
# creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
g = torch.Generator().manual_seed(2147483647) 

# Weights for the trigram model, initialized randomly. 
# These weights can also be set to zero initially as a smoothing technique if desired.
# 27x27x27 weight matrix for trigram probabilities (26 letters + '.')
W = torch.randn((27, 27, 27), generator=g, requires_grad=True)

Now the Gradient descent application

In [ ]:
for k in range(10):
        # === Forward pass to get probabilities for the next character ===
        # This is similar to the bigram model but now we are working with pairs of character indices (ix1, ix2)
        # One-hot encode each of the two input characters separately
        xenc1 = F.one_hot(xs[:, 0], num_classes=27).float()  # (N, 27)
        xenc2 = F.one_hot(xs[:, 1], num_classes=27).float()  # (N, 27)
   

        # Split this step into two parts now that we have two input characters.
        # The first part computes the logits for the next character based on the first character encoding and the weight matrix.
        # The second part contracts the logits with the second character encoding to get the final logits for the next character.

        # For each example, we want W[ix1, ix2, :] as the logits.
        # xenc1 @ W reshaped: (N,27) @ (27, 27*27) -> (N, 27*27), then view as (N, 27, 27)
        # then contract with xenc2: sum over the second character dimension
        logits = (xenc1 @ W.view(27, 27 * 27)).view(-1, 27, 27)  # (N, 27, 27)
        logits = (logits * xenc2.unsqueeze(2)).sum(1)             # (N, 27)

        counts = logits.exp() # convert logits to counts
        probs = counts / counts.sum(1, keepdim=True) # normalize counts to get probabilities
        
        # === Loss computation ===
        # compute the negative log likelihood loss with an added regularization term to prevent overfitting
        loss = -probs[torch.arange(len(ys)), ys].log().mean() + 0.01 * (W**2).mean() 
        
        print(f"step {k}: loss = {loss.item():.4f}")

        # === Backward pass ===
        W.grad = None          # zero out gradients from previous step
        loss.backward()

        # === Update ===
        W.data -= 50 * W.grad

step 0: loss = 2.2324
step 1: loss = 2.2324
step 2: loss = 2.2324
step 3: loss = 2.2324
step 4: loss = 2.2324
step 5: loss = 2.2323
step 6: loss = 2.2323
step 7: loss = 2.2323
step 8: loss = 2.2323
step 9: loss = 2.2323


We find that the loss function is lower than with the Bigram method, which is probably to be expected as there is more context for each prediction - however it takes a lot longer to compute. Now we sample and see if the Trigram model can improve upon the Bigram model.

In [ ]:
for i in range(5):
    out = []

    # start with the '.' character for both positions
    ix1, ix2 = 0, 0 
    while True:
        # one-hot encode each input character index
        xenc1 = F.one_hot(torch.tensor([ix1]), num_classes=27).float()  
        xenc2 = F.one_hot(torch.tensor([ix2]), num_classes=27).float()  
        
        logits = (xenc1 @ W.view(27, 27 * 27)).view(-1, 27, 27)  
        logits = (logits * xenc2.unsqueeze(2)).sum(1)             
        
        counts = logits.exp() # convert logits to counts
        probs = counts / counts.sum(1, keepdim=True) # normalize counts to get probabilities
        
        ix3 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item() # sample the next character index
        out.append(itos[ix3]) # append the corresponding character to the output list
        
        if ix3 == 0: # if we reach the '.' character, we stop generating characters for this name
            break
        
        ix1, ix2 = ix2, ix3 # shift the characters for the next iteration
    
    print(''.join(out)) # join the list of characters to form a name and print

praybge.
tonn.
kens.
jashra.
milmerik.


Overall an improvement on the Bigram method, it generates words which sound a lot closer to something resembling names and even sometimes returns real recognisable names.